IMPORT NECESSARY LIBRARIES


In [8]:
import pandas as pd
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
def load_data(description_file, rating_file):
    # Load descriptions and ratings
    descriptions = pd.read_csv(description_file)
    ratings = pd.read_csv(rating_file)
    
    # Aggregate ratings by place ID
    avg_ratings = ratings.groupby('id')['rating'].mean().reset_index()
    avg_ratings.rename(columns={'id': 'ID', 'rating': 'Average_Rating'}, inplace=True)
    
    # Merge descriptions with average ratings on ID
    data = pd.merge(descriptions, avg_ratings, on='ID', how='inner')
    if 'Description' not in data.columns or 'Average_Rating' not in data.columns:
        raise ValueError("Merged dataset must contain 'Description' and 'Average_Rating' columns.")
    
    data['Description'] = data['Description'].fillna('')  # Handle missing descriptions
    return data

In [10]:
def load_distilbert_model():
    model_name = "distilbert-base-uncased"
    tokenizer = DistilBertTokenizer.from_pretrained(model_name)
    model = DistilBertModel.from_pretrained(model_name).to('cuda')
    return tokenizer, model

# Step 3: Generate Embeddings for Descriptions
def generate_embedding(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True, padding="max_length").to('cuda')
    with torch.no_grad():
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()  # Mean pooling
    return embedding


In [11]:
def generate_all_embeddings(data, tokenizer, model):
    embeddings = []
    for description in data['Description']:
        embedding = generate_embedding(description, tokenizer, model)
        embeddings.append(embedding)
    return embeddings

In [12]:
def recommend_places(query_description, data, embeddings, top_n=5, weight_similarity=0.7, weight_rating=0.3):
    # Generate embedding for the query description
    tokenizer, model = load_distilbert_model()
    query_embedding = generate_embedding(query_description, tokenizer, model).reshape(1, -1)
    
    # Compute cosine similarity between query and all place descriptions
    similarities = cosine_similarity(query_embedding, embeddings).flatten()
    
    # Add similarity scores to the dataset
    data['Similarity_Score'] = similarities
    
    # Compute final recommendation score (weighted combination of similarity and average rating)
    data['Recommendation_Score'] = (weight_similarity * data['Similarity_Score'] +
                                     weight_rating * data['Average_Rating'])
    
    # Sort by recommendation score and return top N places
    recommended_places = data.nlargest(top_n, 'Recommendation_Score')
    
    return recommended_places[['Name', 'Description', 'Average_Rating', 'Similarity_Score', 'Recommendation_Score']]

In [14]:
def main():
    # File paths for descriptions and ratings
    description_file = 'Output/PreparedData.csv'
    rating_file = '../Data/FinalDataset/all_ratings.csv'
    
    # Load merged dataset
    print("Loading data...")
    data = load_data(description_file, rating_file)
    
    # Load DistilBERT model and tokenizer
    print("Loading DistilBERT model...")
    tokenizer, model = load_distilbert_model()
    
    # Generate embeddings for all descriptions
    print("Generating embeddings for all descriptions...")
    embeddings = generate_all_embeddings(data, tokenizer, model)
    
    # Example query description from the user
    query_description = "I want to visit a historical place with cultural significance."
    
    # Recommend places based on query description and user ratings
    print("Recommending places...")
    recommendations = recommend_places(query_description, data, embeddings)
    
    print("\nRecommended Places:")
    print(recommendations)

if __name__ == "__main__":
    main()

Loading data...
Loading DistilBERT model...


AssertionError: Torch not compiled with CUDA enabled

In [15]:
if cuda.is_available():
    print("CUDA is available")

NameError: name 'cuda' is not defined